# Actividad: Evaluación comparativa de arquitecturas convolucionales

Para este notebook se te solicita construir, entrenar y analizar modelos CNN para clasificar imágenes mediante un dataset CIFAR.

**Entregable:** Reporte en la evaluación de la capacidad de arquitectura implementada. Construír arquitecturas propias finalizando con la implementación de una arquitectura clásica mediante transfer learning.


## Toma como base el código visto en clase y desarrolla los siguientes puntos:
- Diseño e implementación de 2 arquitecturas CNN y utilización de una arquitectura de transfer learning.

- Buen uso de data augmentation y regularización.

- Comparación experimental entre arquitecturas y reporte claro (un solo markdown con conclusión sobre la comparación).





In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# Bypass de certificados SSL para descargas en Mac
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# Cargar dataset CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalizar píxeles a rango [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Convertir etiquetas a One-Hot Encoding
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

# Pipeline de Data Augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1166s 7us/step


## Definiciones de modelos

In [3]:
# 1. Arquitectura CNN Propia 1: Básica / Ligera
cnn_1 = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
], name="CNN_Basica")

# 2. Arquitectura CNN Propia 2: Con Regularización (BatchNormalization + Dropout)
cnn_2 = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, (3, 3), padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
], name="CNN_Regularizada")

# 3. Arquitectura Transfer Learning: MobileNetV2
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
base_model.trainable = False  # Congelar capas preentrenadas

transfer_model = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name="MobileNetV2_Transfer")

/opt/anaconda3/envs/redestf/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/var/folders/6l/sz_mpffn23g7khfkrkljlr4w0000gn/T/ipykernel_59487/4222062006.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(32, 32, 3))


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Entrenamiento de modelos.

In [ ]:
epochs = 15
batch_size = 64

# --- Compilar y Entrenar Modelo 1 ---
cnn_1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history_1 = cnn_1.fit(x_train, y_train_cat, epochs=epochs, batch_size=batch_size, validation_data=(x_test, y_test_cat), verbose=1)

# --- Compilar y Entrenar Modelo 2 ---
cnn_2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history_2 = cnn_2.fit(x_train, y_train_cat, epochs=epochs, batch_size=batch_size, validation_data=(x_test, y_test_cat), verbose=1)

# --- Compilar y Entrenar Modelo 3 (Transfer Learning) ---
transfer_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
history_3 = transfer_model.fit(x_train, y_train_cat, epochs=epochs, batch_size=batch_size, validation_data=(x_test, y_test_cat), verbose=1)

Epoch 1/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - accuracy: 0.4041 - loss: 1.6505 - val_accuracy: 0.4962 - val_loss: 1.3779
Epoch 2/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.5125 - loss: 1.3698 - val_accuracy: 0.5579 - val_loss: 1.2909
Epoch 3/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.5500 - loss: 1.2719 - val_accuracy: 0.5863 - val_loss: 1.1904
Epoch 4/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.5782 - loss: 1.1992 - val_accuracy: 0.5971 - val_loss: 1.1595
Epoch 5/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - accuracy: 0.5947 - loss: 1.1512 - val_accuracy: 0.6174 - val_loss: 1.0777
Epoch 6/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.6066 - loss: 1.1187 - val_accuracy: 0.6276 - val_loss: 1.0744
Epoch 7/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.6212 - loss: 1.0831 - val_accuracy: 0.6506 - val_loss: 1.0041
Epoch 8/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.6301 - loss: 1.0610 - 

## Estadística y gráficos

In [ ]:
plt.figure(figsize=(14, 5))

# Gráfico de Precisión (Accuracy)
plt.subplot(1, 2, 1)
plt.plot(history_1.history['val_accuracy'], label='CNN Básica (Val)')
plt.plot(history_2.history['val_accuracy'], label='CNN Regularizada (Val)')
plt.plot(history_3.history['val_accuracy'], label='MobileNetV2 Transfer (Val)')
plt.title('Comparación de Precisión en Validación')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Gráfico de Pérdida (Loss)
plt.subplot(1, 2, 2)
plt.plot(history_1.history['val_loss'], label='CNN Básica (Val)')
plt.plot(history_2.history['val_loss'], label='CNN Regularizada (Val)')
plt.plot(history_3.history['val_loss'], label='MobileNetV2 Transfer (Val)')
plt.title('Comparación de Pérdida en Validación')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Conclusiones.

Escribe tus conclusiones de las arquitecturas hechas ¿Cuál fue el mejor? ¿Por qué? ¿Qué mejoraría? ¿Cómo lo mejoraría?

### Análisis comparativo y conclusiones

1. **¿Cuál fue el mejor modelo y por qué?**
El mejor rendimiento en general lo obtuvo la **CNN Regularizada** (Modelo 2). Aunque MobileNetV2 es una arquitectura avanzada, sus pesos originales fueron entrenados con imágenes de resolución $224 \times 224$, por lo que sufre pérdida de resolución al trabajar con imágenes tan pequeñas como las de CIFAR-10 ($32 \times 32$). Por su parte, la CNN Regularizada incorporó `BatchNormalization` y `Dropout`, evitando el sobreajuste (*overfitting*) que sí sufrió la CNN Básica.

2. **¿Qué mejoraría y cómo lo mejoraría?**
* **Ajuste fino (*Fine-Tuning*) en Transfer Learning:** Descongelar las últimas capas convolucionales de MobileNetV2 y entrenarlas con una tasa de aprendizaje muy baja ($10^{-5}$) para adaptar sus filtros al tamaño nativo de $32 \times 32$.
* **Redimensión de entrada:** Aplicar una capa de reescalado (*UpSampling*) a las imágenes de entrada a $96 \times 96$ o $224 \times 224$ previo al modelo preentrenado.
* **Estrategia de aprendizaje:** Implementar `ReduceLROnPlateau` para reducir el *learning rate* automáticamente cuando la pérdida de validación se estanque.